# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Greemines/Flyrank-Notebook-1/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# **Reproduce the validated Week-6 output**

Week 7 uses the validated Logistic Regression ranking from Week 6 as the starting point for the action playbook.

The Week-6 model used three features: impressions_90d, ctr, and avg_position. The target was defined as trend_direction == "down". Validation used a grouped-by-client split so that no client appeared in both training and test data.

The grouped validation used 23,837 training rows from 25 clients and 6,163 test rows from 7 clients, with zero client overlap. It measured 35% Precision@20 and 44% Precision@50.

This setup reproduces that validated ranking so the Week-7 queue is generated from the same model design rather than creating a new model.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


df = pd.read_csv(
    "/content/content_refresh_anonymized.csv"
)


features = [
    "impressions_90d",
    "ctr",
    "avg_position"
]

df_model = df[
    features
    + [
        "client_id",
        "trend_direction",
        "content_id"
    ]
].copy()


df_model["target"] = (
    df_model["trend_direction"] == "down"
).astype(int)

X = df_model[features]
y = df_model["target"]
groups = df_model["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        X,
        y,
        groups=groups
    )
)

train_df = df_model.iloc[train_idx].copy()
test_df = df_model.iloc[test_idx].copy()

train_clients = set(
    train_df["client_id"]
)

test_clients = set(
    test_df["client_id"]
)

client_overlap = (
    train_clients & test_clients
)

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(client_overlap))

assert len(client_overlap) == 0


X_train = train_df[features].copy()
X_test = test_df[features].copy()

y_train = train_df["target"]
y_test = test_df["target"]

X_train["impressions_90d"] = np.log1p(
    X_train["impressions_90d"]
)

X_test["impressions_90d"] = np.log1p(
    X_test["impressions_90d"]
)

model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "logistic",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

model.fit(
    X_train,
    y_train
)


model_prob = model.predict_proba(
    X_test
)[:, 1]

results = test_df[
    [
        "content_id",
        "client_id",
        "impressions_90d",
        "ctr",
        "avg_position",
        "target",
        "trend_direction"
    ]
].copy()

results["model_score"] = model_prob

model_ranking = (
    results
    .sort_values(
        "model_score",
        ascending=False
    )
    .reset_index(drop=True)
)

model_ranking["model_rank"] = (
    model_ranking.index + 1
)

print("\nValidated model ranking recreated.")

display(
    model_ranking[
        [
            "model_rank",
            "content_id",
            "model_score",
            "target",
            "trend_direction"
        ]
    ].head(20)
)

Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Client overlap: 0

Validated model ranking recreated.


,model_rank,content_id,model_score,target,trend_direction
0,1,content_8c19996aa890,0.796025,1,down
1,2,content_5fe46e04994d,0.795426,1,down
2,3,content_4c36c775b818,0.791912,1,down
3,4,content_db5989a78dd3,0.783450,0,up
4,5,content_9532f197bbc8,0.778049,1,down
5,6,content_73c54f78c06a,0.771596,0,stable
6,7,content_c84a0ab98e90,0.771142,0,stable
7,8,content_cea79ef51519,0.769601,1,down
8,9,content_2db251d1a841,0.768289,0,stable
9,10,content_e12868d1f396,0.762968,0,stable


In [ ]:
def precision_at_k(
    ranking_df,
    k
):
    return ranking_df.head(k)["target"].mean()


precision_20 = precision_at_k(
    model_ranking,
    20
)

precision_50 = precision_at_k(
    model_ranking,
    50
)

print(
    f"Precision@20: {precision_20 * 100:.1f}%"
)

print(
    f"Precision@50: {precision_50 * 100:.1f}%"
)

assert round(precision_20 * 100, 1) == 35.0
assert round(precision_50 * 100, 1) == 44.0

print("\nWeek-6 validation results reproduced: PASS")

Precision@20: 35.0%
Precision@50: 44.0%

Week-6 validation results reproduced: PASS


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

## **1. Ranked actions + reason codes**

The action queue uses the validated Week-6 Logistic Regression ranking as decision-support. Pages near the top of the ranking are candidates for earlier human review.

The grouped-by-client validation measured 35% Precision@20 and 44% Precision@50. The Week-4 baseline measured 30% Precision@20 and 44% Precision@50. The model therefore showed a measured difference at the top 20 pages, while the results were equal at the top 50 pages.

The model also produced false positives. In the top 20, 7 pages were correctly identified as declining and 13 were not declining. This means a high model score should be treated as a review priority rather than as proof that a page is declining.

### **Reason codes**

- `HIGH_SCORE`: page is near the top of the validated model ranking and should be reviewed earlier.
- `SIGNAL_CHECK`: reviewer checks the three model inputs: impressions_90d, ctr, and avg_position.
- `TREND_CHECK`: reviewer checks the observed trend_direction and supporting recent data.
- `REVIEW_CONFLICT`: model ranking and observed evidence do not agree, so additional review is required.
- `MISS_CHECK`: lower-ranked pages are sampled periodically because the model does not identify every declining page.

The queue does not recommend an automatic content change. It identifies pages that look worth reviewing first based on the validated ranking.

In [ ]:

queue = model_ranking.copy()

queue["priority_action"] = "Manual review"

queue["reason_code"] = "HIGH_SCORE"

queue["human_review_required"] = True

queue = queue[
    [
        "model_rank",
        "content_id",
        "client_id",
        "model_score",
        "trend_direction",
        "impressions_90d",
        "ctr",
        "avg_position",
        "priority_action",
        "reason_code",
        "human_review_required"
    ]
].copy()

queue = queue.rename(
    columns={
        "model_rank": "rank"
    }
)

print("Action queue created.")
print("Rows:", len(queue))

display(
    queue.head(20)
)

Action queue created.
Rows: 6163


,rank,content_id,client_id,model_score,trend_direction,impressions_90d,ctr,avg_position,priority_action,reason_code,human_review_required
0,1,content_8c19996aa890,client_4e07408562,0.796025,down,509252,0.15,2.5,Manual review,HIGH_SCORE,True
1,2,content_5fe46e04994d,client_4e07408562,0.795426,down,517715,0.14,4.2,Manual review,HIGH_SCORE,True
2,3,content_4c36c775b818,client_4e07408562,0.791912,down,463103,0.41,2.3,Manual review,HIGH_SCORE,True
3,4,content_db5989a78dd3,client_4e07408562,0.783450,up,345111,0.21,5.4,Manual review,HIGH_SCORE,True
4,5,content_9532f197bbc8,client_4e07408562,0.778049,down,309192,0.87,2.0,Manual review,HIGH_SCORE,True
5,6,content_73c54f78c06a,client_f369cb89fc,0.771596,stable,213963,0.10,4.7,Manual review,HIGH_SCORE,True
6,7,content_c84a0ab98e90,client_f369cb89fc,0.771142,stable,223271,0.03,7.8,Manual review,HIGH_SCORE,True
7,8,content_cea79ef51519,client_f369cb89fc,0.769601,down,208798,0.23,5.2,Manual review,HIGH_SCORE,True
8,9,content_2db251d1a841,client_f369cb89fc,0.768289,stable,198671,0.18,5.6,Manual review,HIGH_SCORE,True
9,10,content_e12868d1f396,client_4e07408562,0.762968,stable,149712,0.07,2.9,Manual review,HIGH_SCORE,True


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## 2. **Intended use and limits**

### **Intended use**

This playbook is intended for a content or SEO team that needs to prioritize pages for human review. The validated Logistic Regression model ranks pages using impressions_90d, ctr, and avg_position.

The highest-ranked pages are review candidates. The ranking helps decide where human attention can start; it does not decide which pages should be changed.

### **Limits**

The grouped-by-client validation measured 35% Precision@20 and 44% Precision@50. The baseline measured 30% Precision@20 and 44% Precision@50. The model therefore showed a modest measured difference at Precision@20 but no difference at Precision@50.

The top-20 error analysis found 7 correctly identified declining pages and 13 false positives. This shows that the queue contains pages that are not declining, so model ranking cannot be treated as a definitive label.

The validation also used a particular dataset, feature set, and validation period. The results do not establish how the model will perform on future data.

The model should therefore be used as directional decision-support for prioritizing review, not as a guarantee of future decline and not as evidence that changing a page will improve performance.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## 3. Human review + the no-go list

Every ranked page requires human review before any content action is taken.

### Human review rules

The reviewer should:

1. Check the model score and queue rank.
2. Check impressions_90d, ctr, and avg_position.
3. Check the observed trend_direction and recent performance context.
4. Look for conflicting evidence between the model ranking and observed page behavior.
5. Decide whether there is a defensible reason to investigate a content issue.
6. Record the final human decision before making a change.

### No-go automation list

The model output must not automatically:

- rewrite or publish content;
- delete or redirect a page;
- change factual claims;
- declare that a page is definitely declining;
- claim that a content refresh will improve performance;
- treat a low model score as proof that a page is healthy;
- make irreversible content or SEO changes.

The model produces a review queue only. A human makes the final content decision.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*



The validated grouped-by-client results provide the current reference point:

- Precision@20: 35%
- Precision@50: 44%
- Baseline Precision@20: 30%
- Baseline Precision@50: 44%

These are measured results for this validation setup, not permanent performance guarantees.

### Monitoring triggers

The playbook should be reviewed when:

1. A later labeled evaluation shows materially lower Precision@20 or Precision@50 than the validated results.
2. The baseline consistently matches or exceeds the model.
3. The distribution or meaning of impressions_90d, ctr, or avg_position changes materially.
4. The definition of a declining page changes.
5. The content population or operating context changes substantially.
6. Human reviewers repeatedly find that high-ranked pages are not useful review candidates.

### Retrain trigger

Retraining should follow evidence that the model's ranking usefulness has degraded or that the underlying data or task has changed. A single incorrect page is not enough to justify retraining.

Monitoring should therefore focus on measured ranking performance and human-review usefulness rather than assuming that a fixed threshold will remain valid indefinitely.`

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

The ranked action queue is exported to `work/outputs/action_queue.csv`.

This queue is generated from the validated Week-6 model ranking and contains the pages prioritized for human review, along with their rank, model score, reason code, and review requirement.

The CSV is a generated artifact for the research paper and is not committed to Git. The notebook regenerates the file when it is executed.

In [ ]:
from pathlib import Path

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

paper_queue = queue[
    [
        "rank",
        "content_id",
        "model_score",
        "priority_action",
        "reason_code",
        "human_review_required"
    ]
].copy()

queue_path = output_dir / "action_queue.csv"

paper_queue.to_csv(
    queue_path,
    index=False
)

print(f"Queue exported to: {queue_path}")
print(f"Rows exported: {len(paper_queue)}")

display(paper_queue.head(20))

Queue exported to: work/outputs/action_queue.csv
Rows exported: 6163


,rank,content_id,model_score,priority_action,reason_code,human_review_required
0,1,content_8c19996aa890,0.796025,Manual review,HIGH_SCORE,True
1,2,content_5fe46e04994d,0.795426,Manual review,HIGH_SCORE,True
2,3,content_4c36c775b818,0.791912,Manual review,HIGH_SCORE,True
3,4,content_db5989a78dd3,0.783450,Manual review,HIGH_SCORE,True
4,5,content_9532f197bbc8,0.778049,Manual review,HIGH_SCORE,True
5,6,content_73c54f78c06a,0.771596,Manual review,HIGH_SCORE,True
6,7,content_c84a0ab98e90,0.771142,Manual review,HIGH_SCORE,True
7,8,content_cea79ef51519,0.769601,Manual review,HIGH_SCORE,True
8,9,content_2db251d1a841,0.768289,Manual review,HIGH_SCORE,True
9,10,content_e12868d1f396,0.762968,Manual review,HIGH_SCORE,True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Demo Outline — CTR Decline Review Ranking

**Runtime: ~5–7 minutes**

1. **The problem (30 sec)**
   Content teams can't manually review every page. Show one stat: 30,000 pages, 32 clients,
   limited reviewer time.

2. **The naive approach and why it's not enough (45 sec)**
   Show the Week-4 rule-based baseline formula. Explain: a single fixed rule struggles because
   no one signal (impressions, CTR, position) reliably separates declining pages on its own.

3. **The honest validation setup (60 sec)**
   Walk through *why* grouped-by-client splitting matters — show the random-split vs.
   grouped-split chart (60% vs. 35% Precision@20) as the "before you trust any number, check
   your split" moment. This is the strongest teaching beat in the demo.

4. **The result (60 sec)**
   Show the model-vs-baseline chart: 35.0% vs. 30.0% at Precision@20, tied at 44.0% at
   Precision@50. Say plainly: modest, directional improvement — not a breakthrough.

5. **Where it breaks (60 sec)**
   Show the top-20 error breakdown: 7 correct, 13 false positives. Be upfront that most
   top-ranked pages are still false leads — this builds credibility, not doubt.

6. **The playbook (60 sec)**
   Show the no-go list. Emphasize: this produces a review queue, not an automated action.
   A human makes every call.

7. **Close (30 sec)**
   One line: "This is decision-support for triage, validated the way it would need to be
   validated to be trusted — not oversold." Point to the live paper URL.

**If asked a hard question, have ready:**
- "Why only 3 features?" → Kept baseline and model identical on purpose, to isolate what the
  model adds beyond the rule.
- "Could this leak?" → Explain the leakage audit found no target-derived leakage in the final
  feature set, but that check wasn't exhaustive across every column.
- "Why not causal?" → Cross-sectional data, no intervention — correlation with an observed
  label only.

##**Social Post**
I built and validated a ranking model to help content teams prioritize which pages to review
for declining search performance — and the most interesting part wasn't the model, it was the
validation.

A naive random train/test split measured 60% precision in the top 20. But 31 clients showed up
in both the training and test sets, which inflates that number. Once I split by client instead
(zero overlap), the honest result was 35% — a modest gain over a simple rule-based baseline
(30%), not a breakthrough.

I'd rather show the real, smaller number than the flattering, wrong one.

Full paper (methodology, results, limitations, and an action playbook for review teams):
https://greemines.github.io/Flyrank-Notebook-1/

Built on the FlyRank ML Internship dataset — flyrank.ai

##**Employer Summary**

I built a ranking model to help content teams prioritize pages for declining-CTR review, using
a 30,000-row anonymized dataset spanning 32 clients. I validated it with a client-grouped train/test
split — deliberately avoiding the more flattering but misleading result a random split would have
shown — and measured a modest, honest improvement over a rule-based baseline (35% vs. 30% precision
in the top 20). The full paper, including limitations and a decision-support playbook for how the
ranking should and shouldn't be used, is live here: https://greemines.github.io/Flyrank-Notebook-1/.